# ENVIRONMENT

In [1]:

! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

# BASICS

In [3]:
import bs4

from langchainhub import Client

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\tongi\AppData\Local\Temp\ipykernel_20132\880883599.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


#### LOAD DOCUMENT 

In [4]:
loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content" , "post_title" , "post-header")
        )
    ),
)

docs = loader.load()


Load the document or website and remove the unnecessary stuff like footer, ads, slidebar etc.

#### SPLIT

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

split = text_splitter.split_documents(docs)

This particular step breaks the documents into the chunks (Small parts)

#### EMBEDING

In [6]:
vectorstore = Chroma.from_documents(
    documents=split,
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

The chunks are then embedded, that is, they form a vector, and that vector is stored in chroma. 

#### PROMPT

In [7]:
from langchain_core.load import loads
hub = Client()
prompt = hub.pull("rlm/rag-prompt")
if isinstance(prompt, str):
    prompt = loads(prompt)

C:\Users\tongi\AppData\Local\Temp\ipykernel_20132\1607537911.py:3: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = hub.pull("rlm/rag-prompt")
c:\Users\tongi\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)
C:\Users\tongi\AppData\Local\Temp\ipykernel_20132\1607537911.py:5: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(prompt)
C:\Users\tongi\AppData\Local\Temp\ipykernel_20132\1607537911.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrus

In this, we use Ready Made RAG prompt. 

#### LLM

In [8]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

In this, we call the OpenAI model and send the request to it . Temperature zero means no creativity gives the most factual, consistent answer every time. Good for RAG because you want accuracy, not randomness.

#### POST PROCESSING

In [9]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In this, we format the doc. It is in the form of a list, but we try to do it in the format such that it is easy to read for the AI Model as a prompt. 

#### CHAIN

In [10]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

This form complete system were the user ask the question and get the responce.

#### QUESTION

In [11]:
response = rag_chain.invoke(
    "What is Task Decomposition?"
)

print(response)


Task Decomposition is a technique used to break down complex tasks into smaller and simpler steps, allowing for easier execution and understanding. It involves transforming big tasks into multiple manageable tasks to enhance model performance. Different approaches, such as simple prompting, task-specific instructions, and reliance on external classical planners, can be used for task decomposition.


This gives us the final response to the question.

## 🔄 RAG Pipeline Flowchart

Below is the complete flow of the Retrieval-Augmented Generation (RAG) system, showcasing both the **Indexing Phase** and the **Retrieval & Generation Phase**:

```mermaid
flowchart TD
    %% Nodes & Styling
    subgraph Indexing ["🛠️ INDEXING PHASE"]
        direction LR
        Load["📄 Load<br/>(WebBaseLoader)"] -->|Raw Docs| Split["✂️ Split<br/>(RecursiveCharacterTextSplitter)"]
        Split -->|Text Chunks| Embed["🧬 Embed<br/>(OpenAIEmbeddings)"]
        Embed -->|Vectors| Store[("🗄️ Store<br/>(Chroma VectorStore)")]
    end

    subgraph Generation ["⚡ RETRIEVAL & GENERATION PHASE"]
        direction LR
        Query["❓ User Query"] --> Retrieve["🔍 Retrieve<br/>(Similarity Search)"]
        Store -.->|Fetch Context| Retrieve
        Retrieve -->|Prompt + Context| Generate["🧠 Generate<br/>(LLM / GPT-3.5-Turbo)"]
        Generate --> Answer["💬 Output Answer"]
    end

    Indexing --> Generation

    %% Colors & Aesthetics
    style Indexing fill:#f4f7f6,stroke:#94a3b8,stroke-width:1px,stroke-dasharray: 5 5;
    style Generation fill:#f8fafc,stroke:#94a3b8,stroke-width:1px,stroke-dasharray: 5 5;
    
    classDef step fill:#ffffff,stroke:#3b82f6,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef store fill:#eff6ff,stroke:#2563eb,stroke-width:2px,color:#1e3a8a,font-weight:bold;
    classDef query fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f,font-weight:bold;
    classDef answer fill:#ecfdf5,stroke:#059669,stroke-width:2px,color:#065f46,font-weight:bold;

    class Load,Split,Embed,Retrieve,Generate step;
    class Store store;
    class Query query;
    class Answer answer;
```


# INDEXING

#### DOCUMENTS


In [12]:
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

#### COUNT TOKEN

In [13]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

8

#### TEXT EMBEDDING

In [14]:
from langchain_openai import OpenAIEmbeddings
embd = OpenAIEmbeddings()
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

1536

#### COSINE SIMILARITY

In [15]:
import numpy as np
def cosine_similarity(vec1,vec2):
    dot_product = np.dot(vec1,vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product/(norm_vec1*norm_vec2)

similarity = cosine_similarity(query_result,document_result)
print(similarity)

0.8806977856520075


#### LOAD DOCUMENT

In [16]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths= ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs= dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

#### SPLITTER

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 300,
    chunk_overlap = 50
)
splits = text_splitter.split_documents(blog_docs)

#### VECTORSTORES

In [18]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits,embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# RETRIEVAL

In [19]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits , embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k":1})

docs = retriever.invoke("What is task documentation?")

len(docs)

1

# GENERATION

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
 
template = """Answer the question based only on the following context:
{context}

Question:{question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion:{question}\n'), additional_kwargs={})])

In [21]:
llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0)

In [22]:
chain = prompt|llm

In [23]:
chain.invoke({"context":docs,"question":"What is task decomposition"})

AIMessage(content='Task decomposition is the process of breaking down a task into smaller subgoals or steps that can be achieved sequentially. It can be done using simple prompting, task-specific instructions, or with human inputs.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 277, 'total_tokens': 316, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000197, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000197, 'upstream_inference_prompt_cost': 0.0001385, 'upstream_inference_completions_cost': 5.85e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1780561801-XXbh5QwGhm2OqEG0xLnP', 'finish_reason': 'stop', 'l

In [24]:
from langchain_core.load import loads
hub = Client()
prompt_hub_rag = hub.pull("rlm/rag-prompt")

C:\Users\tongi\AppData\Local\Temp\ipykernel_20132\1157509328.py:3: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt_hub_rag = hub.pull("rlm/rag-prompt")
c:\Users\tongi\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


In [25]:
prompt_hub_rag

'{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don\'t know the answer, just say that you don\'t know. Use three sentences maximum and keep the answer concise.\\nQuestion: {question} \\nContext: {context} \\nAnswer:", "input_variables": ["question", "context"], "template_format": "f-string"}}}}], "input_variables": ["question", "context"]}}'

#### RAG CHAIN

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'Task Decomposition is a technique used by agents to break down complex tasks into smaller and simpler steps, allowing for better planning and execution. It can be achieved through methods such as Chain of Thought and Tree of Thoughts, which involve breaking down tasks into manageable steps and exploring multiple reasoning possibilities at each step. Task decomposition can also be facilitated through simple prompting, task-specific instructions, or human inputs.'